IMPORTANDO BIBLIOTECAS

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
# Preparando o ambiente para visualizações a partir da primeira analise

df = pd.read_csv("data/superstore.csv",encoding="latin1")

df["Order Date"] = pd.to_datetime(df["Order Date"])
df["Ship Date"] = pd.to_datetime(df["Ship Date"])

df["Year"] = df["Order Date"].dt.year
df["Month"] = df["Order Date"].dt.month

df["Shipping Days"] = (df["Ship Date"] - df["Order Date"]).dt.days

ANALISES

In [ ]:
# Vendas x Lucro x Quantidade por Categoria

category = (df.groupby('Category')
    .agg(
        Sales=('Sales', 'sum'),
        Profit=('Profit', 'sum'),
        Quantity=('Quantity', 'sum')
        )
)

category

In [6]:
# Ordenando os dados por margem

category['Profit Margin'] = category['Profit'] / category['Sales']
category = category.sort_values('Profit Margin', ascending=False)

In [ ]:
#  Vendas x Lucro x Quantidade por Sub-categorias

subcategory = (df.groupby('Sub-Category')
    .agg(
        Sales=('Sales', 'sum'),
        Profit=('Profit', 'sum'),
        Quantity=('Quantity', 'sum')
        )
)

subcategory.sort_values('Profit', ascending=False)
subcategory.sort_values('Profit')


In [ ]:
# Top 5 Sub-categorias por Lucro

top_subcategories = subcategory.sort_values('Profit', ascending=False).head(5)
top_subcategories

In [ ]:
# Top 5 Sub-categorias por Prejuízo

top_loss_subcategories = subcategory.sort_values('Profit', ascending=True).head(5)
top_loss_subcategories

In [ ]:
# Lucro por Sub-categoria

profit_subcategory = df.groupby('Sub-Category')['Profit'].sum().sort_values(ascending=False)

profit_subcategory.plot(kind='barh', figsize=(8, 5), color='mediumseagreen')
plt.title('Lucro por Sub-categoria')
plt.xlabel('Lucro')
plt.ylabel('Sub-categoria')
plt.show()

In [ ]:
# Cruzando Segmento x Categoria

segment_category = (df.groupby(["Segment", "Category"])["Sales"].sum().reset_index())
segment_category

In [ ]:
# Segmento x Categoria

sns.barplot(data=segment_category, x='Segment', y='Sales', hue='Category')
plt.title('Vendas por Segmento e Categoria')
plt.xlabel('Segmento')
plt.ylabel('Vendas')
plt.show()

In [ ]:
# Criando uma tabela dinâmica para cruzar Segmento x Categoria x Vendas

pivot_sales = pd.pivot_table(
    df, 
    values='Sales', 
    index='Segment', 
    columns='Category', 
    aggfunc='sum'
)

pivot_sales

In [ ]:
# Tabela dinâmica de Segmento x Categoria x Lucro

pivot_profit = pd.pivot_table(
    df,
    values='Profit',
    index='Segment',
    columns='Category',
    aggfunc='sum'
)

pivot_profit

In [ ]:
# Região x Categoria

pivot_region_category = pd.pivot_table(
    df, 
    values='Profit', 
    index='Region', 
    columns='Category', 
    aggfunc='sum'
)

pivot_region_category

In [ ]:
# Heatmap de Lucro por Região e Categoria

plt.figure(figsize=(10, 6))
sns.heatmap(
    pivot_region_category, 
    annot=True, 
    fmt=".0f", cmap="YlGnBu"
)

plt.title('Lucro por Região e Categoria')
plt.xlabel('Categoria')
plt.ylabel('Região')
plt.show()

In [ ]:
# Analise por estado

state = (df.groupby('State').agg(
    Sales=('Sales', 'sum'),
    Profit=('Profit', 'sum'),
    Quantity=('Quantity', 'sum')
))

state.sort_values('Profit')

In [ ]:
# Estados com maior prejuízo com vendas acima de 50.000

loss_states = state.loc[
    (state['Profit'] < 0) & (state['Sales'] > 50000)
].sort_values('Profit')
loss_states

In [ ]:
# Query

state.query('Profit < 0 and Sales > 50000').sort_values('Profit')

In [30]:
# Investigando um estado que apresenta prejuízo com vendas acima de 50.000

estado_problema = 'Texas'

texas = df[df["State"] == estado_problema]

texas.groupby('Sub-Category')['Profit'].sum().sort_values()

Sub-Category
Binders       -14705.0738
Appliances     -6147.2225
Furnishings    -3312.6786
Machines       -2666.8434
Chairs         -2515.6490
Bookcases      -2391.1377
Tables         -2216.6766
Supplies        -837.2795
Storage         -763.7054
Fasteners         80.7357
Labels           200.4020
Art              316.3538
Envelopes        848.1760
Accessories     1105.8501
Copiers         1629.9615
Paper           2422.9703
Phones          3222.4608
Name: Profit, dtype: float64

In [ ]:
# Média de desconto por sub-categoria no estado do Texas

texas_analysis = (
    texas.groupby("Sub-Category")
         .agg(
             Sales=("Sales", "sum"),
             Profit=("Profit", "sum"),
             Avg_Discount=("Discount", "mean")
         )
         .sort_values("Profit")
)

texas_analysis

In [ ]:
# Disconto por Categoria

discount_category = (
    df.groupby("Category")
      .agg(
          Avg_Discount=("Discount", "mean"),
          Sales=("Sales", "sum"),
          Profit=("Profit", "sum")
      )
)

discount_category

In [ ]:
# Criando a margem de lucro

discount_category["Profit Margin"] = (discount_category["Profit"] / discount_category["Sales"])

In [ ]:
# A categoria com maior desconto é também a de menor margem de lucro?

discount_category.sort_values("Avg_Discount",ascending=False)

discount_category.sort_values("Avg_Discount",ascending=False)

In [ ]:
# Analise dos clientes

clientes= (df.groupby(["Customer ID", "Customer Name"]).agg(
        Sales=("Sales", "sum"),
        Profit=("Profit", "sum"),
        Orders=("Order ID", "nunique")
    )
)

clientes.head()

In [ ]:
# Ticket Médio

clientes["Average Ticket"] = (clientes["Sales"] / clientes["Orders"])

clientes.sort_values("Average Ticket", ascending=False).head(10)

In [ ]:
# Top Clierntes por Sales

clientes.sort_values("Average Ticket", ascending=False).head(10)

In [ ]:
# Top Clientes por Lucro

clientes.sort_values("Profit", ascending=False).head(10)

In [ ]:
# Clientes que geram prejuízo

piores_clientes = clientes[clientes["Profit"] < 0]

piores_clientes.sort_values("Profit").head(10)

In [ ]:
nome_do_cliente = (piores_clientes.sort_values("Profit").index[0])

clientes[clientes["Profit"] < 0].sort_values("Profit").head(10)

In [61]:
# Análise dos clientes que mais geram prejuízo

piores_clientes.shape[0]
clientes.shape[0]

piores_clientes_rate = (len(piores_clientes) /len(clientes))

print(f"{piores_clientes_rate:.2%}")

19.55%


In [63]:
# Ticket Médio Geral

total_vendas = df["Sales"].sum()
total_pedidos = df["Order ID"].nunique()

average_ticket = (total_vendas / total_pedidos)

print(f"${average_ticket:,.2f}")

$458.61


In [ ]:
# Pedidos por cliente

pedidos_por_cliente = df.groupby("Customer ID")["Order ID"].nunique().sort_values(ascending=False)
pedidos_por_cliente.describe()

In [ ]:
pedidos_por_cliente.sort_values(ascending=False).head(10)

In [ ]:
# Analise de vendas por ano e categoria

anos_categoria = (df.groupby(["Year", "Category"])["Sales"].sum().reset_index())

anos_categoria 

In [ ]:
# Visualização da evolução das vendas por categoria ao longo dos anos

sns.lineplot(
    data=anos_categoria,
    x="Year",
    y="Sales",
    hue="Category",
    marker="o"
)

plt.title("Sales Evolution by Category")
plt.show()

In [ ]:
# Crescimento percentual entre os anos

ano_sales = (df.groupby("Year")["Sales"].sum())
crescimento = ano_sales.pct_change()

crescimento

In [ ]:
ano_categoria = (df.groupby(["Year", "Category"])["Sales"].sum().reset_index())
ano_categoria["crscimento"] = ano_categoria.groupby("Category")["Sales"].pct_change()

ano_categoria